# LLM-as-a-Judge Pipeline

Clasificación de preguntas SOCE como **ACUSATORIA / NO_ACUSATORIA** usando few-shot prompting con GPT-4o-mini.

## 1. Imports y configuración

In [13]:
%pip install tenacity

^C
Note: you may need to restart the kernel to use updated packages.


In [ ]:
%pip install dotenv

  Using cached dotenv-0.9.9-py2.py3-none-any.whl.metadata (279 bytes)
  Using cached python_dotenv-1.2.2-py3-none-any.whl.metadata (27 kB)
Using cached dotenv-0.9.9-py2.py3-none-any.whl (1.9 kB)
Using cached python_dotenv-1.2.2-py3-none-any.whl (22 kB)

   ---------------------------------------- 0/2 [python-dotenv]
   ---------------------------------------- 2/2 [dotenv]

Note: you may need to restart the kernel to use updated packages.


In [1]:
import os
import asyncio
import random
import hashlib
import json
import time
from pathlib import Path

import pandas as pd
import numpy as np
from openai import AsyncOpenAI, RateLimitError
from tenacity import retry, stop_after_attempt, wait_exponential, retry_if_exception_type
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, f1_score, recall_score, precision_score,
    classification_report, confusion_matrix
)
from dotenv import load_dotenv, find_dotenv
import matplotlib.pyplot as plt
import seaborn as sns

_env_path = find_dotenv(usecwd=True)
if not _env_path:
    raise FileNotFoundError(
        "No se encontró .env. Crea llm_judge_project/.env con OPENAI_API_KEY=sk-..."
    )
load_dotenv(_env_path)

PROJECT_ROOT = Path(_env_path).parent
client       = AsyncOpenAI(api_key=os.getenv("OPENAI_API_KEY"))

SEMAPHORE    = asyncio.Semaphore(50)
RANDOM_STATE = 42
MODEL        = "gpt-4o-mini"

DATA_RAW     = PROJECT_ROOT / "llm_judge_project" / "data" / "raw"
DATA_SPLITS  = PROJECT_ROOT / "llm_judge_project" / "data" / "splits"
DATA_RESULTS = PROJECT_ROOT / "llm_judge_project" / "data" / "results"
ITER_LOGS    = DATA_RESULTS / "iteration_logs"
FINAL_EVAL   = DATA_RESULTS / "final_evaluation"
PROMPTS_DIR  = PROJECT_ROOT / "llm_judge_project" / "prompts"
CACHE_FILE   = DATA_RESULTS / "cache" / "response_cache.json"

response_cache: dict = {}
if CACHE_FILE.exists():
    with open(CACHE_FILE, encoding='utf-8') as _f:
        response_cache = json.load(_f)
    print(f"Caché cargada: {len(response_cache)} entradas desde {CACHE_FILE.name}")

print(f"PROJECT_ROOT: {PROJECT_ROOT.resolve()}")
print(f"API key presente: {bool(os.getenv('OPENAI_API_KEY'))}")
print(f"Concurrencia: {SEMAPHORE._value} requests simultáneas")

Caché cargada: 59 entradas desde response_cache.json
PROJECT_ROOT: C:\Users\martin\Documents\proyecto_integrador\entregable_2-3
API key presente: True
Concurrencia: 50 requests simultáneas


## 2. Carga y exploración del dataset

In [2]:
# Detectar archivo automáticamente (xlsx o csv)
xlsx_files = list(DATA_RAW.glob("*.xlsx"))
csv_files  = list(DATA_RAW.glob("*.csv"))

if xlsx_files:
    df_raw = pd.read_excel(xlsx_files[0])
    print(f"Dataset cargado: {xlsx_files[0].name}")
elif csv_files:
    df_raw = pd.read_csv(csv_files[0])
    print(f"Dataset cargado: {csv_files[0].name}")
else:
    raise FileNotFoundError(f"No hay dataset en {DATA_RAW}")

# Renombrar columna de etiqueta si viene del dataset Roh López
if 'final_pregunta_isAcusatoria' in df_raw.columns:
    df_raw = df_raw.rename(columns={'final_pregunta_isAcusatoria': 'etiqueta'})

assert 'pregunta' in df_raw.columns, "Falta columna 'pregunta'"
assert 'etiqueta' in df_raw.columns, "Falta columna 'etiqueta'"
df_raw['etiqueta'] = df_raw['etiqueta'].astype(int)

print(f"\nTotal preguntas: {len(df_raw)}")
print(f"\nDistribución de clases:")
dist = df_raw['etiqueta'].value_counts(normalize=True).rename({0: 'No acusatoria', 1: 'Acusatoria'})
print(dist.round(4))
display(df_raw[['pregunta', 'etiqueta']].head())

Dataset cargado: dataset_roh_5005.xlsx

Total preguntas: 5005

Distribución de clases:
etiqueta
No acusatoria    0.9706
Acusatoria       0.0294
Name: proportion, dtype: float64


,pregunta,etiqueta
0,Por favor requerimos se entregue un diagrama d...,0
1,Indique de manera detallada y de manera taxati...,0
2,4.En el punto 6 Información que dispone la ent...,0
3,Se puede ofertar un equipo que utilice cubetas...,0
4,PODRIANN SUBIR UNA IMAGEN DEL SPAGUETTI ESPIRA...,0


## 3. Partición estratificada

Split fijo con `random_state=42`. **No modificar** una vez generado.

In [3]:
# 80/20 global
train_80, test_20 = train_test_split(
    df_raw, test_size=0.20, random_state=RANDOM_STATE, stratify=df_raw['etiqueta']
)

# 75/25 dentro del 80%
train_inner, val_inner = train_test_split(
    train_80, test_size=0.25, random_state=RANDOM_STATE, stratify=train_80['etiqueta']
)

DATA_SPLITS.mkdir(parents=True, exist_ok=True)
train_80.to_csv(DATA_SPLITS / "train_80.csv",       index=False)
test_20.to_csv(DATA_SPLITS / "test_20.csv",         index=False)
train_inner.to_csv(DATA_SPLITS / "train_inner.csv", index=False)
val_inner.to_csv(DATA_SPLITS / "val_inner.csv",     index=False)

for name, df in [("train_80", train_80), ("test_20", test_20),
                 ("train_inner", train_inner), ("val_inner", val_inner)]:
    pct = df['etiqueta'].mean() * 100
    print(f"{name:<12}: {len(df):>5} preguntas | acusatorias: {df['etiqueta'].sum():>3} ({pct:.2f}%)")

train_80    :  4004 preguntas | acusatorias: 118 (2.95%)
test_20     :  1001 preguntas | acusatorias:  29 (2.90%)
train_inner :  3003 preguntas | acusatorias:  89 (2.96%)
val_inner   :  1001 preguntas | acusatorias:  29 (2.90%)


## 4. Carga del prompt template

In [4]:
prompt_path = PROMPTS_DIR / "prompt_v1.txt"
with open(prompt_path, encoding='utf-8') as f:
    prompt_template = f.read()

print(f"Prompt: {prompt_path.name}  ({len(prompt_template)} caracteres)\n")
print(prompt_template)

Prompt: prompt_v1.txt  (1712 caracteres)

Eres un experto en contratación pública del Ecuador. Clasifica preguntas del SOCE como ACUSATORIA o NO_ACUSATORIA.

ACUSATORIA: imputa intención dolosa a la entidad — afirma que un requisito fue diseñado para favorecer a alguien, denuncia corrupción consumada, o insinúa que la entidad actúa en beneficio de intereses particulares.

NO ACUSATORIA: consulta, cuestiona o pide cambios en requisitos sin imputar mala fe, aunque use lenguaje enérgico o cite la ley.

CRITERIO: ¿el oferente afirma o insinúa que la entidad actuó con DOLO o MALA FE?
→ SÍ: ACUSATORIA | NO: NO_ACUSATORIA

DISTINCIONES CLAVE:
- Condicional ("podría", "no sea que", "se estaría") → NO ACUSATORIA
- Afirmativo ("están favoreciendo", "esto es ilegal y lo hacen") → ACUSATORIA
- Citar ley para pedir cambio → NO ACUSATORIA
- Señalar que la entidad viola la ley a sabiendas → ACUSATORIA
- Acusar a otros oferentes (no a la entidad) → NO ACUSATORIA
- Señalar comportamiento pasado indebid

## 5. Funciones utilitarias

In [5]:
def sample_few_shot_examples(train_df: pd.DataFrame, n: int = 4, seed: int = 42):
    """Muestrea n ejemplos acusatorios y n no acusatorios de train_df."""
    acus    = train_df[train_df['etiqueta'] == 1].sample(n=n, random_state=seed)['pregunta'].tolist()
    no_acus = train_df[train_df['etiqueta'] == 0].sample(n=n, random_state=seed)['pregunta'].tolist()
    return acus, no_acus


def build_prompt(template: str, acus: list, no_acus: list, pregunta: str) -> str:
    """Rellena los placeholders del template con los ejemplos y la pregunta."""
    p = template
    for i, ej in enumerate(acus, 1):
        p = p.replace(f'{{ejemplo_acusatoria_{i}}}', ej)
    for i, ej in enumerate(no_acus, 1):
        p = p.replace(f'{{ejemplo_no_acusatoria_{i}}}', ej)
    return p.replace('{pregunta}', pregunta)


def parse_response(text: str) -> int:
    """Convierte respuesta del LLM a etiqueta binaria. -1 si no parseable."""
    t = (text or '').strip().upper()
    if 'NO_ACUSATORIA' in t or 'NO ACUSATORIA' in t:
        return 0
    if 'ACUSATORIA' in t:
        return 1
    return -1


print("Funciones utilitarias definidas.")

Funciones utilitarias definidas.


## 6. Cliente LLM asíncrono

In [6]:
@retry(
    retry=retry_if_exception_type(RateLimitError),
    stop=stop_after_attempt(6),
    wait=wait_exponential(multiplier=1, min=2, max=60),
    reraise=True
)
async def classify_question(prompt: str) -> str:
    cache_key = hashlib.md5(f'{MODEL}:{prompt}'.encode()).hexdigest()
    if cache_key in response_cache:
        return response_cache[cache_key]

    async with SEMAPHORE:
        response = await client.chat.completions.create(
            model=MODEL,
            messages=[{'role': 'user', 'content': prompt}],
            temperature=0,
            max_tokens=10
        )
    result = response.choices[0].message.content or ''
    response_cache[cache_key] = result
    return result


def save_cache():
    CACHE_FILE.parent.mkdir(parents=True, exist_ok=True)
    with open(CACHE_FILE, 'w', encoding='utf-8') as f:
        json.dump(response_cache, f, ensure_ascii=False)


async def classify_batch(prompts: list[str]) -> list[str]:
    """Clasifica prompts en paralelo con asyncio.gather + semáforo."""
    total = len(prompts)
    print(f"Clasificando {total} preguntas con {MODEL}...")

    pbar = tqdm(total=total, desc='Clasificando', unit='preg')

    async def _one(p: str) -> str:
        result = await classify_question(p)
        pbar.update(1)
        return result

    results = await asyncio.gather(*(_one(p) for p in prompts))
    pbar.close()

    save_cache()
    print(f"Completado. Caché: {len(response_cache)} entradas.")
    return list(results)


print("Cliente LLM asíncrono definido.")

Cliente LLM asíncrono definido.


## 7. Función `run_iteration`

In [7]:
async def run_iteration(iteration_num: int, prompt_template: str,
                        train_df: pd.DataFrame, val_df: pd.DataFrame,
                        seed: int = 42) -> dict:
    t0 = time.time()
    ITER_LOGS.mkdir(parents=True, exist_ok=True)

    # 1. Few-shot
    acus_ex, no_acus_ex = sample_few_shot_examples(train_df, n=4, seed=seed)

    # 2. Construir prompts
    prompts = [
        build_prompt(prompt_template, acus_ex, no_acus_ex, q)
        for q in val_df['pregunta'].tolist()
    ]

    # 3. Clasificar
    raw_responses = await classify_batch(prompts)

    # 4. Parsear
    predictions = [parse_response(r) for r in raw_responses]

    # 5. DataFrame de resultados
    results_df = val_df[['pregunta', 'etiqueta']].copy().reset_index(drop=True)
    results_df['respuesta_llm'] = raw_responses
    results_df['prediccion']    = predictions
    results_df['correcta']      = results_df['etiqueta'] == results_df['prediccion']

    # 6. Métricas sobre predicciones válidas
    valid      = results_df[results_df['prediccion'] != -1]
    unparseable = int((results_df['prediccion'] == -1).sum())
    y_true = valid['etiqueta'].values
    y_pred = valid['prediccion'].values

    metrics = {
        'iteration':   iteration_num,
        'f1':          f1_score(y_true, y_pred, zero_division=0),
        'recall':      recall_score(y_true, y_pred, zero_division=0),
        'precision':   precision_score(y_true, y_pred, zero_division=0),
        'accuracy':    accuracy_score(y_true, y_pred),
        'fn':          int(((valid['etiqueta'] == 1) & (valid['prediccion'] == 0)).sum()),
        'fp':          int(((valid['etiqueta'] == 0) & (valid['prediccion'] == 1)).sum()),
        'unparseable': unparseable,
        'elapsed_s':   round(time.time() - t0, 1),
        'seed':        seed,
    }

    # 7. Guardar
    tag = f"iter_{iteration_num:02d}"
    results_df.to_csv(ITER_LOGS / f"{tag}_preds.csv", index=False)

    log_file = ITER_LOGS / "iterations_log.csv"
    summary = pd.DataFrame([metrics])
    if log_file.exists():
        existing = pd.read_csv(log_file)
        existing = existing[existing['iteration'] != iteration_num]
        summary = pd.concat([existing, summary], ignore_index=True)
    summary.to_csv(log_file, index=False)

    # 8. Imprimir
    print(f"\n{'='*55}")
    print(f"Iteración {iteration_num}  ({metrics['elapsed_s']}s)")
    print(f"  F1:        {metrics['f1']:.4f}")
    print(f"  Recall:    {metrics['recall']:.4f}")
    print(f"  Precision: {metrics['precision']:.4f}")
    print(f"  Accuracy:  {metrics['accuracy']:.4f}")
    print(f"  FN: {metrics['fn']}  FP: {metrics['fp']}  Sin parsear: {metrics['unparseable']}")
    print(f"{'='*55}\n")
    return metrics


print("run_iteration definida.")

run_iteration definida.


## 8. Iteraciones de ajuste del prompt

Corre la celda de la iteración actual, analiza los errores (sección 9), edita el prompt en `prompts/`, actualiza el número y vuelve a correr.

**Criterios de parada:** F1 ≥ 0.90 | 3 iteraciones sin mejora | máx 15 iteraciones.

In [8]:
# ── Iteración 1 ──────────────────────────────────────────────────────────────
prompt_path = PROMPTS_DIR / "prompt_v1.txt"
with open(prompt_path, encoding='utf-8') as f:
    prompt_template = f.read()

metrics_1 = await run_iteration(
    iteration_num=1,
    prompt_template=prompt_template,
    train_df=train_inner,
    val_df=val_inner,
    seed=42
)

Clasificando 1001 preguntas con gpt-4o-mini...


Clasificando:   4%|▎         | 36/1001 [01:18<1:11:40,  4.46s/preg]

CancelledError: 

In [ ]:
# ── Iteración N ──────────────────────────────────────────────────────────────
# Edita prompt_vN.txt con las mejoras identificadas en el análisis de errores,
# luego actualiza el nombre y el número de iteración.

prompt_path = PROMPTS_DIR / "prompt_v2.txt"   # <-- cambiar versión
with open(prompt_path, encoding='utf-8') as f:
    prompt_template = f.read()

metrics_N = await run_iteration(
    iteration_num=2,              # <-- incrementar
    prompt_template=prompt_template,
    train_df=train_inner,
    val_df=val_inner,
    seed=42
)

## 9. Análisis de errores

Identifica patrones en FN/FP para guiar el refinamiento del prompt.

In [ ]:
def analyze_errors(iteration_num: int, max_examples: int = 10):
    preds_file = ITER_LOGS / f"iter_{iteration_num:02d}_preds.csv"
    if not preds_file.exists():
        print(f"No encontrado: {preds_file}")
        return None, None

    df = pd.read_csv(preds_file)
    fn_df = df[(df['etiqueta'] == 1) & (df['prediccion'] == 0)].reset_index(drop=True)
    fp_df = df[(df['etiqueta'] == 0) & (df['prediccion'] == 1)].reset_index(drop=True)

    print(f"{'='*60}")
    print(f"Iteración {iteration_num} — Análisis de errores")
    print(f"  Falsos Negativos (acusatoria → NO detectada): {len(fn_df)}")
    print(f"  Falsos Positivos (no acusatoria → marcada):   {len(fp_df)}")
    print(f"{'='*60}")

    if not fn_df.empty:
        print(f"\n── FALSOS NEGATIVOS (primeros {min(max_examples, len(fn_df))}) ──")
        for _, row in fn_df.head(max_examples).iterrows():
            print(f"\n  LLM: [{row.get('respuesta_llm','?')}]")
            print(f"  {row['pregunta'][:300]}")

    if not fp_df.empty:
        print(f"\n── FALSOS POSITIVOS (primeros {min(max_examples, len(fp_df))}) ──")
        for _, row in fp_df.head(max_examples).iterrows():
            print(f"\n  LLM: [{row.get('respuesta_llm','?')}]")
            print(f"  {row['pregunta'][:300]}")

    return fn_df, fp_df


# Cambiar el número según la iteración a analizar
fn_df, fp_df = analyze_errors(1)

## 10. Evolución de métricas

In [ ]:
log_file = ITER_LOGS / "iterations_log.csv"

if log_file.exists():
    metrics_df = pd.read_csv(log_file).sort_values('iteration')
    display(metrics_df[['iteration', 'f1', 'recall', 'precision', 'accuracy',
                         'fn', 'fp', 'elapsed_s']].round(4).reset_index(drop=True))

    fig, ax = plt.subplots(figsize=(10, 5))
    for col, label, ls in [('f1', 'F1', '-'), ('recall', 'Recall', '--'), ('precision', 'Precisión', ':')]:
        ax.plot(metrics_df['iteration'], metrics_df[col], marker='o', label=label, linestyle=ls)
    ax.axhline(0.90, color='red', linestyle='--', alpha=0.5, label='Umbral F1=0.90')
    ax.set_xlabel('Iteración')
    ax.set_ylabel('Métrica')
    ax.set_title('Evolución de métricas por iteración')
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print("Aún no hay iteraciones registradas.")

## 11. Evaluación final

> **Ejecutar una sola vez** después de congelar el prompt definitivo.

Actualiza `prompt_final_path` con el nombre de tu mejor prompt antes de correr.

In [ ]:
FINAL_EVAL.mkdir(parents=True, exist_ok=True)
test_df = pd.read_csv(DATA_SPLITS / "test_20.csv")

prompt_final_path = PROMPTS_DIR / "prompt_v1.txt"   # <-- mejor prompt
with open(prompt_final_path, encoding='utf-8') as f:
    prompt_final = f.read()

best_seed = 42
acus_ex, no_acus_ex = sample_few_shot_examples(train_inner, n=4, seed=best_seed)
prompts_test = [
    build_prompt(prompt_final, acus_ex, no_acus_ex, q)
    for q in test_df['pregunta'].tolist()
]

raw_test = await classify_batch(prompts_test)
test_df = test_df.copy().reset_index(drop=True)
test_df['respuesta_llm'] = raw_test
test_df['prediccion']    = [parse_response(r) for r in raw_test]

valid_test = test_df[test_df['prediccion'] != -1]
y_true_t   = valid_test['etiqueta'].values
y_pred_t   = valid_test['prediccion'].values

print("\n" + "="*60)
print("EVALUACIÓN FINAL — TEST_20")
print("="*60)
print(classification_report(y_true_t, y_pred_t, target_names=['No acusatoria', 'Acusatoria']))

final_metrics = {
    'f1':        f1_score(y_true_t, y_pred_t, zero_division=0),
    'recall':    recall_score(y_true_t, y_pred_t, zero_division=0),
    'precision': precision_score(y_true_t, y_pred_t, zero_division=0),
    'accuracy':  accuracy_score(y_true_t, y_pred_t),
}
test_df.to_csv(FINAL_EVAL / "final_preds.csv", index=False)
pd.DataFrame([final_metrics]).to_csv(FINAL_EVAL / "final_metrics.csv", index=False)
print(f"Resultados guardados en {FINAL_EVAL}")

cm = confusion_matrix(y_true_t, y_pred_t)
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['No acusatoria', 'Acusatoria'],
            yticklabels=['No acusatoria', 'Acusatoria'], ax=ax)
ax.set_ylabel('Etiqueta real')
ax.set_xlabel('Predicción LLM')
ax.set_title('Matriz de confusión — Evaluación final')
plt.tight_layout()
plt.show()

## 12. Comparación con ensemble del proyecto

Completa los valores `None` con los reportados en el documento de tesis.

In [ ]:
comparison = pd.DataFrame([
    {'Modelo': 'LLM Judge (prompt final)',  'F1': final_metrics['f1'],        'Recall': final_metrics['recall'],    'Precision': final_metrics['precision'], 'Accuracy': final_metrics['accuracy']},
    {'Modelo': 'Ensemble (M1+M2+M3)',       'F1': None, 'Recall': None, 'Precision': None, 'Accuracy': None},
    {'Modelo': 'Logistic Regression (M1)',  'F1': None, 'Recall': None, 'Precision': None, 'Accuracy': None},
    {'Modelo': 'Naive Bayes (M2)',          'F1': None, 'Recall': None, 'Precision': None, 'Accuracy': None},
    {'Modelo': 'Random Forest (M3)',        'F1': None, 'Recall': None, 'Precision': None, 'Accuracy': None},
])
display(comparison.round(4))